In [ ]:
# LLM API 설정
LLM_API_URL = 'http://192.168.0.2:1234/v1/chat/completions'  # 필요시 변경
LLM_MODEL = 'local-llm'  # 필요시 변경

import pandas as pd
import numpy as np
import re
import csv
import time
import os
import requests
from tqdm import tqdm

def LoadLLM(txt, extxt, src_lang, tgt_lang):
    if not txt or txt == extxt:
        return extxt
    url = LLM_API_URL
    headers = {'Content-Type': 'application/json'}
    # 리그 오브 레전드 e스포츠 팬게임 스타일 가이드
    style_guide = (
        "You are translating for a League of Legends esports fan game. "
        "Use official League of Legends and esports terminology. "
        "Keep the tone competitive, energetic, and suitable for esports commentary or game dialogue. "
        "If there are special esports terms, use the standard translation used in the League of Legends community. "
        "Maintain consistency with previous translations and the official esports style."
    )
    prompt = (
        f"{style_guide}\n"
        f"Translate the following text from {src_lang} to {tgt_lang}. Only return the translated text.\n\nText: {txt}"
    )
    data = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }
    try:
        response = requests.post(url, headers=headers, json=data, timeout=30)
        response.raise_for_status()
        result = response.json()
        return result['choices'][0]['message']['content'].strip()
    except Exception as e:
        print(f"번역 오류: {e}")
        return txt

def createFolder(directory):
    try:
        if not os.path.exists(directory):
            os.makedirs(directory)
    except OSError:
        print ('Error: Creating directory. ' +  directory)

def Convert(loadList, language, languageFull, replaceList, src_lang, tgt_lang, currentVersion):
    # 체크리스트 불러오기
    if os.path.isfile('./checkList.csv'):
        checkList = pd.read_csv('./checkList.csv', encoding = 'utf-8')
    else:
        checkList = pd.DataFrame(columns=['Language', 'File'])

    for loadFile in loadList:
        checkLanguage = False
        # 체크리스트
        for i in range(len(checkList)):
            if  checkList['Language'][i] == languageFull:
                if checkList['File'][i] == loadFile:
                    if str(checkList['Version'][i]) == currentVersion:
                        print(languageFull + ' ' + loadFile)
                        checkLanguage = True
                        break

        if checkLanguage == False:
            #데이터 불러오기
            originRead = pd.read_csv('./English/' + loadFile + '.csv', encoding = 'utf-8')
            current_read = originRead.copy()

            # 바꿀 데이터인지 확인
            isReplace = False
            for replaceData in replaceList:
                if loadFile == replaceData:
                    isReplace = True
                    break

            # 파일이 있어야 비교
            if os.path.isfile('./BeforeEnglish/' + loadFile + '.csv') and isReplace == False:
                before_read = pd.read_csv('./BeforeEnglish/' + loadFile + '.csv', encoding = 'utf-8')
                # 데이터프레임 병합하여 차이점 표시
                df_diff = originRead.merge(before_read, how='outer', indicator=True)
                result = df_diff[df_diff['_merge'] == 'left_only'].drop(columns=['_merge'])

                if not result.empty:
                    colList = ['Name', 'Dec']
                    exTxt = ''
                    exEx = ''
                    for col in colList:
                        for dfCol in result.columns:
                            if col in dfCol:
                                for r in result.index:
                                    if exEx == current_read.at[r, col]:
                                        result.at[r, col] = exTxt
                                    else:
                                        van = LoadLLM(result.at[r, col], exTxt, src_lang, tgt_lang)
                                        exEx = current_read.at[r, col]
                                        result.at[r, col] = van
                                        exTxt = van

                    languageRead = pd.read_csv('./'+ languageFull +'/' + loadFile + '.csv', encoding = 'utf-8')
                    setColList = ['ID', 'Name', 'Dec']
                    for _index in result.index:
                        for col in setColList:
                            for dfCol in languageRead.columns:
                                if col in dfCol:
                                    languageRead.at[_index, dfCol] = result.at[_index, col]
                                    break
                    languageRead = languageRead.iloc[:len(originRead)]
                    if loadFile == 'Etc':
                        languageRead['Korean'] = originRead['Korean'] 
                    createFolder('./' + languageFull)
                    languageRead.to_csv('./'+ languageFull +'/' + loadFile + '.csv', mode='w', index=False, encoding='utf-8-sig')
            else:
                colList = ['Name', 'Dec']
                exTxt = ''
                exEx = ''
                for col in colList:
                    for dfCol in current_read.columns:
                        if col in dfCol:
                            for r in tqdm(current_read.index):
                                if exEx == current_read.at[r, col]:
                                    current_read.at[r, col] = exTxt
                                else:
                                    van = LoadLLM(current_read.at[r, col], exTxt, src_lang, tgt_lang)
                                    exEx = current_read.at[r, col]
                                    current_read.at[r, col] = van
                                    exTxt = van
                createFolder('./' + languageFull)
                current_read.to_csv('./'+ languageFull +'/' + loadFile + '.csv', mode='w', index=False, encoding='utf-8-sig')

            new_data = pd.DataFrame({'Language': [languageFull], 'File': [loadFile], 'Version': [currentVersion]})
            checkList = pd.concat([checkList, new_data], ignore_index=True)
            checkList.to_csv('./checkList.csv', mode='w', index=False, encoding='utf-8-sig')
            print(languageFull + ' ' + loadFile)

def CsvNRemove(loadFile, languageFull):
    file_path = './'+ languageFull +'/' + loadFile + '.csv'
    with open(file_path, 'r', newline='', encoding='utf-8') as infile:
        reader = csv.reader(infile)
        rows = [
            [cell.replace('\n', ' ').replace('\r', ' ').replace('(남성)', ' ') for cell in row]
            for row in reader
        ]
    with open(file_path, 'w', newline='', encoding='utf-8') as outfile:
        writer = csv.writer(outfile)
        writer.writerows(rows)

#불러올 데이터들
loadList = ['Etc', 'Script', 'Fish', 'Achievement']

#완전히 새로운 데이터로 변경
replaceList = []

#현재 버전
currentVersion = '1.31'

# 언어 정보
readLanDF = pd.read_csv('./LanguageList.csv', encoding = 'utf-8')
originLanguageList = ['en','ko','zh-CN','zh-TW','de','fr','es','it','pt','tr','ru','ja','vi','ms','th','id','jw','bn','hi','ar']
languageList = ['ja', 'zh-CN', 'zh-TW', 'vi', 'de', 'ru', 'es', 'ar', 'it', 'ms', 'th', 'tr', 'fr', 'id', 'jw', 'bn', 'hi', 'pt']

# 번역 실행
for lan in range((len(readLanDF) - len(languageList)), len(readLanDF)):
    src_lang = 'English'
    tgt_lang = readLanDF['Language'][lan]
    Convert(loadList, languageList[lan], tgt_lang, replaceList, src_lang, tgt_lang, currentVersion)

for loadFile in loadList:
    originRead = pd.read_csv('./English/' + loadFile + '.csv', encoding = 'utf-8')
    createFolder('./BeforeEnglish/')
    originRead.to_csv('./BeforeEnglish/' + loadFile + '.csv', mode='w', index=False, encoding='utf-8-sig')